# Basic Formulary Tutor

Baseline before RAG: a plain LLM call, no retrieval, no formulary lookup. Same shape as `ai-tutor-rag-system/notebooks/01-Basic_Tutor.ipynb`, re-themed for health insurance/formulary questions and pointed at the local Ollama model this project already uses (see `../../config/config.yaml`).

`langchain_03_table_aware.ipynb` builds the full retrieval pipeline (PDF table extraction, embeddings, FAISS). This notebook is the pre-RAG baseline it improves on: the model answering from its own knowledge only, with a domain-restricted system prompt and no LangChain dependency — just raw HTTP calls to Ollama.

# Setup

In [1]:
# Load the LLM model name from the project's shared config (no API key needed — local Ollama server).
import yaml
from pathlib import Path

config_path = Path("../../config/config.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = config["llm"]["model"]
TEMPERATURE = config["llm"]["temperature"]
MAX_TOKENS = config["llm"]["max_tokens"]

print(f"✅ Config loaded. Model: {MODEL} (temperature={TEMPERATURE}, max_tokens={MAX_TOKENS})")

✅ Config loaded. Model: llama3.1 (temperature=0.3, max_tokens=1000)


# Load the API client

In [2]:
import requests

# No SDK/framework — talk to the local Ollama server directly over its HTTP API.
def call_ollama(messages):
    payload = {
        "model": MODEL,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": TEMPERATURE,
            "num_predict": MAX_TOKENS,
        },
    }
    response = requests.post(OLLAMA_URL, json=payload, timeout=120)
    response.raise_for_status()
    return response.json()["message"]["content"].strip()

print("✅ Ollama client ready")

✅ Ollama client ready


# Query the API

In [3]:
# Define two questions: 1) Related to health insurance/formulary, 2) Unrelated topic.
# These questions will be used to evaluate the model's performance.
QUESTION_FORMULARY = "What does a Tier 1 drug formulary status mean, and how does it usually affect what a member pays?"
QUESTION_NOT_FORMULARY = (
    "What is the name of the highest mountain in the world and its height?"
)

In [4]:
def ask_formulary_tutor(question):
    try:
        # System instructions for the assistant, specialized in health insurance/formulary questions
        instructions = (
            "You are an assistant specialized in answering health insurance and drug formulary-related "
            "questions (plan tiers, coverage rules, requirements/limits, enrollment). "
            "Only answer questions on those topics, else say that you cannot answer this question."
        )

        messages = [
            {"role": "system", "content": instructions},
            {
                "role": "user",
                "content": f"Please provide an informative and accurate answer to the following question.\nQuestion: {question}\nAnswer:",
            },
        ]

        return call_ollama(messages)

    except Exception as e:
        return f"An error occurred: {e}"

In [5]:
# Ask the formulary-related question.
RES_FORMULARY = ask_formulary_tutor(QUESTION_FORMULARY)
print(RES_FORMULARY)

A Tier 1 drug formulary status typically indicates that the medication is a preferred or generic option, often with the lowest copayment or coinsurance amount. Members usually pay the lowest out-of-pocket cost for Tier 1 medications, as they are considered the most cost-effective choice.

In general, the cost-sharing for Tier 1 medications is as follows:

* Copayment: A fixed amount, usually $5-$15 per prescription, which is a flat fee that the member pays for each prescription.
* Coinsurance: A percentage of the medication's cost, often 10% or less, which is a percentage of the total cost that the member pays.

Keep in mind that the specific cost-sharing amounts may vary depending on the health insurance plan, the member's coverage, and any applicable discounts or promotions.


In [6]:
# Ask the unrelated question.
RES_NOT_FORMULARY = ask_formulary_tutor(QUESTION_NOT_FORMULARY)
print(RES_NOT_FORMULARY)

I'm not able to answer this question. I can only provide information on health insurance and drug formulary-related topics.


# History

In [7]:
messages = [
    {
        "role": "system",
        "content": (
            "You are an assistant specialized in answering health insurance and drug formulary-related "
            "questions (plan tiers, coverage rules, requirements/limits, enrollment). "
            "Only answer questions on those topics, else say that you cannot answer this question."
        ),
    },
    {
        "role": "user",
        "content": f"Please provide an informative and accurate answer to the following question.\nQuestion: {QUESTION_FORMULARY}\nAnswer:",
    },
    {"role": "assistant", "content": RES_FORMULARY},
    {
        "role": "user",
        "content": f"Please provide an informative and accurate answer to the following question.\nQuestion: {QUESTION_NOT_FORMULARY}\nAnswer:",
    },
    {"role": "assistant", "content": RES_NOT_FORMULARY},
    {
        "role": "user",
        "content": "Please provide an informative and accurate answer to the following question.\nQuestion: Which formulary tier did you mention in your first answer, and what does that tier generally imply about cost?\nAnswer:",
    },
]

print(call_ollama(messages))

In my previous answer, I mentioned Tier 1. A Tier 1 drug formulary status generally implies that the medication is a preferred or generic option, and it is usually associated with the lowest cost-sharing amount. This typically means that the member will pay the lowest copayment or coinsurance amount for Tier 1 medications, making them the most cost-effective choice.
